# exp_5 — Experiment 5: Model Interpretability Using Explainable AI (XAI)

Source-based notebook converted from the uploaded experiment code document.


**Note:** The uploaded source assumes a trained VGG16 model is already available as `model`. Run the transfer-learning setup from Experiment 4A first, and make the trained VGG16 model available under that variable name. Update the image path in the source cell to your local image.


In [ ]:
# ===========================
# Experiment: Explainable AI (XAI)
# ===========================

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2

from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image

# Check that the CNN model from Experiment 1 is available
print("Model loaded from previous experiment.")
model.summary()

In [ ]:
import os

print("Cats:")
print(os.listdir(r"D:\DL DATASET\dogs_vs_cats_subset\dogs_vs_cats_subset\val\cats")[:10])

print("\nDogs:")
print(os.listdir(r"D:\DL DATASET\dogs_vs_cats_subset\dogs_vs_cats_subset\val\dogs")[:10])

In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np
import matplotlib.pyplot as plt

IMG_SIZE = (224,224)

img_path = r"D:\DL DATASET\dogs_vs_cats_subset\dogs_vs_cats_subset\val\cats\cat.12000.jpg"

img = image.load_img(img_path, target_size=IMG_SIZE)

img_array = image.img_to_array(img)
img_array = img_array / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)

confidence = prediction[0][0]

if confidence > 0.5:
    label = "Dog"
    confidence = confidence
else:
    label = "Cat"
    confidence = 1 - confidence

plt.figure(figsize=(5,5))
plt.imshow(img)
plt.title(f"Prediction: {label}\nConfidence: {confidence:.2%}")
plt.axis("off")
plt.show()

In [ ]:
from tensorflow.keras.models import Model

# Select early, middle and deep convolution layers
layer_outputs = [
    model.get_layer("block1_conv1").output,
    model.get_layer("block3_conv1").output,
    model.get_layer("block5_conv3").output
]

layer_names = [
    "Block1 Conv1",
    "Block3 Conv1",
    "Block5 Conv3"
]

feature_model = Model(
    inputs=model.input,
    outputs=layer_outputs
)

# Generate feature maps
feature_maps = feature_model.predict(img_array)

In [ ]:
import matplotlib.pyplot as plt

for layer_name, feature_map in zip(layer_names, feature_maps):

    plt.figure(figsize=(14, 7))

    plt.suptitle(layer_name, fontsize=16)

    # Display first 16 feature maps
    for i in range(16):
        plt.subplot(4, 4, i + 1)
        plt.imshow(feature_map[0, :, :, i], cmap='viridis')
        plt.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Last convolutional layer in VGG16
last_conv_layer = model.get_layer("block5_conv3")

# Model that returns both conv outputs and final prediction
grad_model = tf.keras.models.Model(
    inputs=model.inputs,
    outputs=[last_conv_layer.output, model.output]
)

with tf.GradientTape() as tape:

    conv_outputs, predictions = grad_model(img_array)

    loss = predictions[:, 0]

# Compute gradients
grads = tape.gradient(loss, conv_outputs)

# Global Average Pooling
pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

# Feature maps
conv_outputs = conv_outputs[0]

# Weighted feature maps
heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
heatmap = tf.squeeze(heatmap)

# Normalize
heatmap = tf.maximum(heatmap, 0)
heatmap /= tf.reduce_max(heatmap)

heatmap = heatmap.numpy()

plt.figure(figsize=(6,6))
plt.imshow(heatmap, cmap='jet')
plt.title("Grad-CAM Heatmap")
plt.colorbar()
plt.axis("off")
plt.show()

In [ ]:
# Original image
original = cv2.imread(img_path)
original = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)

# Resize heatmap
heatmap = cv2.resize(heatmap, (original.shape[1], original.shape[0]))

# Convert to 0-255
heatmap = np.uint8(255 * heatmap)

# Apply color map
heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

# Overlay
superimposed = cv2.addWeighted(original, 0.6, heatmap, 0.4, 0)

plt.figure(figsize=(8,8))
plt.imshow(superimposed)
plt.title("Grad-CAM Overlay")
plt.axis("off")
plt.show()